In [ ]:
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 2 Lab — Data Wrangling
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Detecting and fixing wrong data types
- Identifying and handling missing values
- Finding and correcting invalid values
- Removing duplicates
- Identifying which columns to keep as features

**Estimated time:** 75–90 minutes

---

## The Dataset

You are working with a customer orders extract from a retail company's CRM system. The data team has warned you it has not been cleaned. Your job this week is to audit and fix it before it goes near a model.

Run the cell below to load it.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(7)
n = 300

# --- Base data ---
order_dates = pd.date_range('2024-01-01', '2024-12-31', periods=n)
regions     = np.random.choice(['Northeast', 'northeast', 'NORTHEAST',
                                 'West', 'west', 'South', 'Midwest'], n)
products    = np.random.choice(['Laptop Stand', 'Monitor Arm',
                                 'Keyboard Tray', 'Cable Manager'], n)
units       = np.random.randint(-2, 25, n)          # some negatives
unit_price  = np.random.choice([49.95, 79.95, 34.95, 24.95], n)
discount    = np.random.choice([0, 0.05, 0.10, 0.15, 0.20, 1.50, -0.05], n,
                                p=[0.4, 0.2, 0.15, 0.1, 0.08, 0.04, 0.03])  # invalid discounts
revenue     = np.round(units * unit_price * (1 - np.clip(discount, 0, 1)), 2).astype(float)
customer_id = np.random.randint(1000, 2000, n).astype(float)  # stored as float
rep_notes   = np.random.choice(['Good lead', 'Follow up', 'Closed', '', np.nan], n)

# Introduce missing values
rev_null_idx  = np.random.choice(n, 18, replace=False)
reg_null_idx  = np.random.choice(n, 9,  replace=False)
revenue[rev_null_idx] = np.nan

df_raw = pd.DataFrame({
    'order_id':    range(3001, 3001 + n),
    'order_date':  order_dates.strftime('%Y-%m-%d'),  # stored as string
    'customer_id': customer_id,
    'region':      regions,
    'product':     products,
    'units_sold':  units,
    'unit_price':  unit_price,
    'discount_pct': discount,
    'revenue':     revenue,
    'rep_notes':   rep_notes,
})

# Inject region nulls
df_raw.loc[reg_null_idx, 'region'] = np.nan

# Add 10 exact duplicate rows
duplicates = df_raw.sample(10, random_state=7)
df_raw = pd.concat([df_raw, duplicates], ignore_index=True)

print(f"Dataset loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

---
## Part 1 — The Wrangling Audit

Before fixing anything, run the full audit. You need to know the complete picture of problems before deciding how to handle each one.

Run each cell in sequence and read the output before moving on.

In [ ]:
# Step 1: Shape and types
print("Shape:", df_raw.shape)
print()
df_raw.info()

In [ ]:
# Step 2: Missing values — count and percentage
missing_count = df_raw.isnull().sum()
missing_pct   = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)

audit = pd.DataFrame({'missing': missing_count, 'pct': missing_pct})
print(audit[audit['missing'] > 0])

In [ ]:
# Step 3: Duplicates
print(f"Duplicate rows: {df_raw.duplicated().sum()}")

In [ ]:
# Step 4: Numeric distributions — spot invalid values
df_raw.describe()

In [ ]:
# Step 5: Categorical value counts — spot inconsistencies
print("Region values:")
print(df_raw['region'].value_counts(dropna=False))
print()
print("Unique values per column:")
print(df_raw.nunique())

---
### 💼 Business Context — Audit Before You Fix

The audit reveals four problems in this dataset: wrong types (`order_date` as string, `customer_id` as float), missing values in `revenue` and `region`, invalid values in `discount_pct` and `units_sold`, inconsistent `region` strings, and 10 duplicate rows. Fixing these in the wrong order creates new problems — for example, standardizing region strings before dropping duplicates means duplicates might not match exactly. Work top to bottom: types → duplicates → invalid values → missing values → features.

---

### ✏️ Exercise 1.1 — Document the Problems

Before writing any fix, fill in the table below as a comment. List every problem you found in the audit, which column it affects, and how you plan to handle it. One row per problem.

| Problem | Column(s) | Planned fix |
|:--------|:----------|:------------|
| ... | ... | ... |

In [ ]:
# Exercise 1.1 — document your findings as comments
#
# Problem                    | Column(s)      | Planned fix
# ---------------------------+----------------+----------------------------
# 1.                         |                |
# 2.                         |                |
# 3.                         |                |
# 4.                         |                |
# 5.                         |                |
# 6.                         |                |

---
## Part 2 — Fixing Types and Duplicates

Start with a clean copy of the raw data. Fix types first — many downstream operations depend on having the right types in place.

In [ ]:
# Work on a copy — never modify df_raw directly
df = df_raw.copy()

# Fix order_date: string → datetime, then extract numeric features
df['order_date']    = pd.to_datetime(df['order_date'])
df['order_month']   = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter

# Fix customer_id: float → nullable integer
df['customer_id'] = df['customer_id'].astype('Int64')

# Drop duplicate rows
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Rows before: {before} | After dedup: {len(df)} | Removed: {before - len(df)}")

# Verify types
print()
print(df[['order_date', 'customer_id', 'order_month', 'order_quarter']].dtypes)

---
### 🤖 ML Connection — Why Types Come First

scikit-learn requires numeric input. Fixing types now means every downstream step — encoding, scaling, model fitting — works without runtime errors. Converting `order_date` to datetime also unlocks `order_month` and `order_quarter` as features: two columns that capture seasonal patterns a raw date string cannot provide.

---

### ✏️ Exercise 2.1 — Standardize the Region Column

The `region` column contains `'Northeast'`, `'northeast'`, `'NORTHEAST'`, and `'West'`, `'west'` etc. Standardize all values to title case in one operation. Then confirm the unique values look correct.

In [ ]:
# Exercise 2.1 — standardize region strings
# Hint: chain .str.strip() and .str.title()

df['region'] = 

# Confirm — should show only: Northeast, West, South, Midwest (+ NaN)
print(df['region'].value_counts(dropna=False))

---
## Part 3 — Invalid Values

Two columns have invalid values identified in the audit: `discount_pct` (values above 1.0 or below 0) and `units_sold` (negative values). Fix both.

In [ ]:
# Detect invalid discounts
bad_discount = df[(df['discount_pct'] > 1.0) | (df['discount_pct'] < 0)]
print(f"Invalid discount rows: {len(bad_discount)}")
print(bad_discount[['order_id', 'discount_pct', 'revenue']].head())

print()

# Detect negative units_sold
neg_units = df[df['units_sold'] < 0]
print(f"Negative units_sold rows: {len(neg_units)}")
print(neg_units[['order_id', 'units_sold', 'revenue']].head())

---
### 💼 Business Context — Drop or Fix?

A discount of 150% is not a rounding error — it is a data entry mistake or a system bug. You have two options: cap the value at 1.0 (assuming it was meant to be 100%), or drop the row. Capping is appropriate when the record is otherwise valid and you have reason to believe the extreme value is a known error type. Dropping is appropriate when the entire record is suspect. Here, we cap discounts between 0 and 1 and remove rows with negative units — a negative unit count has no plausible business interpretation.

---

### ✏️ Exercise 3.1 — Fix Invalid Values

1. Cap `discount_pct` so all values fall between 0 and 1 inclusive. Values below 0 become 0; values above 1 become 1. Use `np.clip()` or `.loc[]`.
2. Remove all rows where `units_sold` is less than 0.
3. After both fixes, print the min and max of each column to confirm the ranges are valid.

In [ ]:
# Exercise 3.1

# 1. Cap discount_pct between 0 and 1
df['discount_pct'] = 

# 2. Remove rows with negative units_sold
before = len(df)
df = 
print(f"Rows removed (negative units): {before - len(df)}")

# 3. Verify ranges
print()
print("discount_pct — min:", df['discount_pct'].min(), "| max:", df['discount_pct'].max())
print("units_sold   — min:", df['units_sold'].min(),   "| max:", df['units_sold'].max())

---
## Part 4 — Missing Values

Two columns have missing values: `revenue` (~6%) and `region` (~3%). Handle each separately — the right strategy depends on the column.

In [ ]:
# Current missing value counts after prior fixes
print(df[['revenue', 'region', 'rep_notes']].isnull().sum())

In [ ]:
# Revenue: flag missingness, then fill with median
df['revenue_missing'] = df['revenue'].isnull().astype(int)

revenue_median = df['revenue'].median()
df['revenue']  = df['revenue'].fillna(revenue_median)

print(f"Revenue median used for imputation: ${revenue_median:.2f}")
print(f"Revenue nulls remaining: {df['revenue'].isnull().sum()}")
print(f"revenue_missing flag — value counts:")
print(df['revenue_missing'].value_counts())

---
### 🤖 ML Connection — The Flag-and-Fill Pattern

The `revenue_missing` column tells the model which rows had imputed revenue. If missingness is correlated with something the model is trying to predict — for example, missing revenue might be more common for cancelled orders — this flag is itself a predictive signal. Without it, that information is lost. One extra column costs almost nothing; losing a predictive signal can meaningfully reduce model accuracy.

---

### ✏️ Exercise 4.1 — Handle Remaining Missing Values

1. Fill missing `region` values with the mode (most common region). Print the mode before filling.
2. The `rep_notes` column has many missing values and is free text. Decide whether to keep or drop it, and explain your reasoning in a comment.
3. Run a final missing value check across the whole DataFrame. The only acceptable nulls at this point are in `order_date` (we keep the original) and any columns you intentionally dropped.

In [ ]:
# Exercise 4.1

# 1. Fill missing region with mode
region_mode = 
print(f"Region mode: {region_mode}")
df['region'] = 

# 2. Decision on rep_notes — keep or drop?
# Reasoning:
# your code here (drop or keep)

# 3. Final missing value check
print("\nFinal missing value counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## Part 5 — Feature Identification

With the data clean, decide which columns to keep as features for modelling. The target variable will be defined in Week 3. For now, identify which columns are candidates and which should be dropped — and why.

In [ ]:
# Review what we have after cleaning
print("Columns:", df.columns.tolist())
print()
print("Unique values per column:")
print(df.nunique())
print()
print("Shape:", df.shape)

### ✏️ Exercise 5.1 — Select and Drop Features

Review the column list above. For each column, decide: keep as a feature, keep as the target (we'll predict whether revenue is above the median), or drop.

1. Fill in the table below as a comment.
2. Write the code to drop the columns you identified as drop.
3. Print the final column list and shape.

| Column | Keep / Target / Drop | Reason |
|:-------|:---------------------|:-------|
| `order_id` | | |
| `order_date` | | |
| `customer_id` | | |
| `region` | | |
| `product` | | |
| `units_sold` | | |
| `unit_price` | | |
| `discount_pct` | | |
| `revenue` | | |
| `order_month` | | |
| `order_quarter` | | |
| `revenue_missing` | | |

In [ ]:
# Exercise 5.1

# Columns to drop (list your reasoning as a comment beside each)
cols_to_drop = [
    # 'col_name',   # reason
]

df_clean = df.drop(columns=cols_to_drop)

print("Final columns:", df_clean.columns.tolist())
print("Final shape:",   df_clean.shape)

---
## Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

You receive a second dataset: employee expense reports from the same company, covering the same 12-month period. The data team says it also "has not been cleaned."

| Column | Description |
|:-------|:------------|
| `expense_id` | Unique identifier |
| `employee_id` | Float (same issue as customer_id above) |
| `submission_date` | String (YYYY/MM/DD format this time) |
| `category` | `'travel'`, `'TRAVEL'`, `'meals'`, `'Meals'`, `'equipment'` |
| `amount` | Some negatives, some above $10,000 |
| `approved` | `'Y'`, `'N'`, `'yes'`, `'no'`, `1`, `0` mixed |
| `manager_comments` | Free text, 60% missing |

Build the dataset with `np.random.seed(22)` and 200 rows, then:

1. Run the full wrangling audit.
2. Fix all type problems, including `approved` — standardize it to a boolean column.
3. Handle the invalid `amount` values. Note that expenses above $5,000 require a second approval — they are valid but should be flagged.
4. Handle missing values for each column with a justified strategy.
5. Identify features for a model that predicts whether an expense will be approved.
6. **Reflection:** The `submission_date` uses `YYYY/MM/DD` format instead of `YYYY-MM-DD`. Does `pd.to_datetime()` handle this automatically? Test it and report what you find.

In [ ]:
# Challenge — build the expense dataset
np.random.seed(22)
n_exp = 200

expense = pd.DataFrame({
    'expense_id':       range(8001, 8001 + n_exp),
    'employee_id':      np.random.randint(200, 400, n_exp).astype(float),
    'submission_date':  pd.date_range('2024-01-01', '2024-12-31', periods=n_exp).strftime('%Y/%m/%d'),
    'category':         np.random.choice(['travel', 'TRAVEL', 'meals', 'Meals', 'equipment'], n_exp),
    'amount':           np.round(np.random.uniform(-50, 12000, n_exp), 2),
    'approved':         np.random.choice(['Y', 'N', 'yes', 'no', 1, 0], n_exp),
    'manager_comments': np.random.choice(['Approved', 'Needs receipt', 'Escalated', np.nan], n_exp,
                                          p=[0.2, 0.15, 0.05, 0.60]),
})

# Your wrangling below


---
## Week 2 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| Wrangling order | Types → duplicates → invalid values → missing values → features |
| `df.info()` | Shows types and null counts together — run it first, every time |
| `pd.to_datetime()` | Converts string dates — extract month, quarter, day of week afterward |
| `'Int64'` (nullable int) | Use for integer columns that contain NaN |
| `str.strip().str.title()` | One-line fix for inconsistent category strings |
| `np.clip()` | Cap values to a valid range without a loop |
| Flag-and-fill | Create a binary missing indicator before imputing — preserves signal |
| `fillna(median)` | Preferred for skewed numeric columns |
| `fillna(mode()[0])` | Standard approach for categorical columns |
| Feature identification | Drop IDs, free text, high-null columns, and leakage before modelling |

---
**Next week:** Feature engineering — encoding categorical variables, scaling numeric features, and splitting into training and test sets. This is the final preparation step before the first model.